# Spark UI Deep Dive: Forensic Analysis of Jobs, Stages, and Tasks

**Module**: 10_performance_and_optimization  
**Notebook**: 01_spark_ui_analysis.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Mastering the Spark UI: Jobs, Stages, Storage, Environment, Executors, and SQL tabs, event timelines, and diagnostic triage.
- **Why does this topic exist?**: The Spark UI is your primary window into distributed execution. 90% of tuning decisions originate from UI metrics.
- **What problem does it solve?**: Trying to optimize jobs blindly without knowing if CPU, network, disk, or GC is the bottleneck.
- **Where is it used in real-world Data Engineering?**: Resolving enterprise cluster SLA breaches and performance incidents.

---

## 2. Navigating the Spark UI
- **Jobs Tab**: High-level timeline of triggered actions.
- **Stages Tab**: Deeper metrics per stage: Duration percentiles (Min, 25%, Median, 75%, Max), Shuffle Read/Write bytes, Spill (Memory/Disk).
- **Storage Tab**: Cached DataFrames/RDDs, memory used, percentage cached in RAM.
- **Executors Tab**: Active cores, memory used per executor, GC pause time, failed tasks.
- **SQL Tab**: Visual DAG plan with physical operators and row counters.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Spark_UI_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
res = orders.groupBy("channel").count()
res.show()
print("Open http://localhost:4040/ to inspect the live Spark UI!")

---

## Experiment: Correlating Spark UI Event Timeline with Code

- **1. What are we testing?**: Run a multi-stage join query and inspect Event Timeline bars.
- **2. What do we expect?**: Timeline shows blue (compute), orange (fetch wait), and red (GC).

In [ ]:
products = spark.read.option("header", True).csv("/opt/spark-data/retail/products.csv")
orders.join(products, "product_id").groupBy("category").count().show()
print("Check http://localhost:4040/stages/ for Stage task timelines!")

**3. What happened?**: Spark UI tracked execution timeline across all executor cores.  
**4. Why did it happen?**: LiveListenerBus streams task events to the embedded web server.  
**5. Practical Engineering Lesson**: Large gaps or long bars in the timeline pinpoint the exact execution bottleneck.

---

## Practical Exercises

### Exercise 1 (Beginner): Check UI Port
**Problem Statement**: What port hosts the driver UI?  
**Input / Context**: Port

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Identify Straggler Task
**Problem Statement**: How to spot stragglers in Stages tab.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Executor GC Ratio
**Problem Statement**: Calculate GC percentage threshold.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check UI Port):
```python
print('Port 4040 (Driver) and Port 18080 (History Server)')
```

#### Solution 2 (Identify Straggler Task):
```python
print('Compare Median vs Max task duration. If Max >> Median, it is a straggler.')
```

#### Solution 3 (Executor GC Ratio):
```python
print('If Task GC time exceeds 10% of total duration, investigate JVM memory allocation.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **UI Forensic Flow**: Jobs -> Stages -> Task Summary Table -> SQL Plan.
- **Straggler Detection**: Max task time >> 75th percentile.

### Essential Technical Terms
- `LiveListenerBus`: Internal event queue feeding UI.
- `Event Timeline`: Visual timeline of task threads.

### Common Engineering Mistakes
- **Closing browser and losing metrics**: Use History Server (18080) for completed jobs.

### Interview Questions on This Topic
- **Q: What does an orange segment in the Spark UI Task Timeline indicate?**
  - *A*: Shuffle Fetch Wait time (waiting for remote shuffle map output blocks over the network).

### What to Study Next
Open 02_explain_and_execution_plans.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')